# Interactive dendrogram (single patient)

Interactive dendrogram threshold slider for one correlation network.

**Legacy notebooks merged:**
- TEST_interactive_single_patient.ipynb

In [ ]:
%matplotlib inline
from lrgsglib.config.funcs import move_to_rootf
move_to_rootf(pathname="lrg_eegfc")
from lrg_eegfc.notebook import *

In [ ]:
import networkx as nx
from ipywidgets import interact, FloatSlider
from lrgsglib.core import compute_laplacian_properties, compute_normalized_linkage, compute_optimal_threshold
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import dendrogram

patient = list_patients(Path('data/stereoeeg_patients'))[0]
phase = PHASE_LABELS[0]
band = BRAIN_BANDS_NAMES[2]

corr = load_corr_matrix(patient, phase, band, filter_type='abs', zero_diagonal=True)
if corr is None:
    corr_res = compute_corr_matrix(patient, phase, band, filter_type='abs', zero_diagonal=True, filter_time=5000)
    corr = corr_res.adjacency_matrix

G = nx.from_numpy_array(corr)

spect, L, rho, Trho, tau = compute_laplacian_properties(G, tau=None)
dists = squareform(Trho)
lnkgM, label_list, _ = compute_normalized_linkage(dists, G, method='ward')
clTh, *_ = compute_optimal_threshold(lnkgM, scaling_factor=0.98)


def plot_dendro(scale):
    fig, ax = plt.subplots(figsize=(8, 4))
    _ = dendrogram(lnkgM, ax=ax, color_threshold=scale)
    ax.axhline(scale, color='red', linestyle='--')
    ax.set_yscale('log')
    ax.set_title('Interactive dendrogram')
    plt.show()

slider = FloatSlider(value=clTh, min=clTh * 0.5, max=clTh * 1.5, step=clTh * 0.05)
interact(plot_dendro, scale=slider)